In [0]:
-- ============================================================
-- FILE    : 02_silver_schema.sql
-- LAYER   : Silver
-- SCHEMA  : customer360_silver
-- PURPOSE : CREATE TABLE statements for all 10 Silver tables
--           Snowflake schema — 7 dimensions + 3 facts
--           Tables are rebuilt on every pipeline run (overwrite)
-- ============================================================

CREATE SCHEMA IF NOT EXISTS customer360_silver
COMMENT 'Customer 360 — Silver layer (Snowflake schema)';

-- ────────────────────────────────────────────────────────────
-- DIMENSION TABLES
-- ────────────────────────────────────────────────────────────

-- TABLE : customer360_silver.dim_date
-- GRAIN : One row per calendar date (2023-01-01 → 2026-12-31)
-- NOTE  : Generated programmatically — no source table needed.
--         Pre-built so fact tables can join on date_key string.
CREATE TABLE IF NOT EXISTS customer360_silver.dim_date (
    date_key              STRING   NOT NULL  COMMENT 'PK — ISO date string YYYY-MM-DD e.g. 2024-03-15',
    full_date             DATE               COMMENT 'Full date as DATE type',
    day_of_month          INT                COMMENT '1-31',
    day_of_week           INT                COMMENT '1=Monday 7=Sunday',
    day_name              STRING             COMMENT 'Monday / Tuesday ... Sunday',
    week_number           INT                COMMENT 'ISO week number 1-53',
    month                 INT                COMMENT '1-12',
    month_name            STRING             COMMENT 'January / February ... December',
    quarter               INT                COMMENT '1-4',
    quarter_label         STRING             COMMENT 'Q1 / Q2 / Q3 / Q4',
    year                  INT                COMMENT 'Calendar year e.g. 2024',
    fiscal_quarter_label  STRING             COMMENT 'Combined label e.g. 2024-Q1',
    is_weekend            INT                COMMENT '1 = Saturday or Sunday  0 = weekday',
    is_holiday            INT                COMMENT '1 = Indian public holiday  0 = normal day',
    is_non_working_day    INT                COMMENT '1 = weekend or holiday  0 = working day'
)
USING DELTA
COMMENT 'Date dimension — full spine 2023-2026 with fiscal and holiday flags';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.dim_city
-- GRAIN : One row per city (10 rows total)
-- NOTE  : Snowflake level 1 — dim_location FKs into this.
--         Avoids repeating state/region in every location row.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.dim_city (
    city_key   STRING  NOT NULL  COMMENT 'PK — CITY_ + first 3 chars of city name e.g. CITY_MUM',
    city_name  STRING            COMMENT 'Full city name e.g. Mumbai',
    state      STRING            COMMENT 'Indian state e.g. Maharashtra',
    region     STRING            COMMENT 'North / South / East / West',
    city_tier  STRING            COMMENT 'Tier 1 / Tier 2 — based on population and commerce'
)
USING DELTA
COMMENT 'City master — 10 Indian cities with state, region, tier classification';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.dim_location
-- GRAIN : One row per pincode zone (3 per city = 30 rows total)
-- NOTE  : Snowflake level 2 — FKs into dim_city.
--         fact_transactions and fact_sessions FK into this.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.dim_location (
    location_key  STRING  NOT NULL  COMMENT 'PK — LOC_ + city prefix + sequence e.g. LOC_MUM_01',
    city_key      STRING            COMMENT 'FK → dim_city.city_key',
    city_name     STRING            COMMENT 'Denormalised city name for convenience',
    pincode       STRING            COMMENT 'Synthetic 6-digit Indian pincode',
    zone          STRING            COMMENT 'North Zone / South Zone / East Zone / West Zone / Central Zone'
)
USING DELTA
COMMENT 'Location dimension — 30 locations (3 per city) snowflaked into dim_city';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.dim_category
-- GRAIN : One row per product category (7 rows total)
-- NOTE  : Snowflake level 1 — dim_product FKs into this.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.dim_category (
    category_key   STRING  NOT NULL  COMMENT 'PK — CAT_ + first 4 chars of category e.g. CAT_ELEC',
    category_name  STRING            COMMENT 'Electronics / Fashion / Groceries / Travel / Entertainment / Health / Sports',
    department     STRING            COMMENT 'Technology / Apparel / Food / Services / Media / Wellness / Active Life',
    super_category STRING            COMMENT 'Hard Goods / Soft Goods / Consumables / Experiences'
)
USING DELTA
COMMENT 'Category master — 7 product categories with department and super-category hierarchy';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.dim_product
-- GRAIN : One row per distinct product_id from Bronze
-- NOTE  : Snowflake level 2 — FKs into dim_category.
--         Built from distinct product_ids in bronze_transactions.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.dim_product (
    product_key    STRING  NOT NULL  COMMENT 'PK — same as product_id from Bronze e.g. PRD1234',
    category_key   STRING            COMMENT 'FK → dim_category.category_key',
    category_name  STRING            COMMENT 'Denormalised category name for convenience',
    brand          STRING            COMMENT 'Brand name — varies by category e.g. Samsung / Zara / Nike',
    price_tier     STRING            COMMENT 'Budget / Mid-range / Premium / Luxury',
    base_price     DOUBLE            COMMENT 'Synthetic base price in INR — skewed distribution 50-50000'
)
USING DELTA
COMMENT 'Product dimension — one row per distinct product from Bronze transactions';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.dim_payment_method
-- GRAIN : One row per payment method (5 rows total)
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.dim_payment_method (
    payment_key     STRING  NOT NULL  COMMENT 'PK — PAY_ + method slug e.g. PAY_CREDIT_C',
    method_name     STRING            COMMENT 'Credit Card / Debit Card / UPI / Net Banking / Wallet',
    provider        STRING            COMMENT 'Visa/Mastercard / NPCI / Bank / Paytm/PhonePe',
    payment_type    STRING            COMMENT 'Card / Digital',
    is_digital      INT               COMMENT '1 = digital payment  0 = card payment',
    is_emi_eligible INT               COMMENT '1 = EMI available  0 = no EMI'
)
USING DELTA
COMMENT 'Payment method master — 5 methods with type and EMI eligibility';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.dim_customer
-- GRAIN : One row per customer (latest record wins — deduped on user_id)
-- NOTE  : SCD Type 0 — no history tracked.
--         customer_key = CUST_ + user_id (surrogate key pattern).
--         customer_age_days recomputed on every pipeline run.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.dim_customer (
    customer_key       STRING  NOT NULL  COMMENT 'PK — surrogate key CUST_ + user_id e.g. CUST_A1B2C3D4',
    user_id            STRING            COMMENT 'NK — natural key from Bronze',
    name               STRING            COMMENT 'Full name',
    age                INT               COMMENT 'Age at signup',
    gender             STRING            COMMENT 'Male / Female / Other',
    signup_date        DATE              COMMENT 'Date of account creation',
    customer_age_days  INT               COMMENT 'Days since signup — recomputed every run',
    city               STRING            COMMENT 'City at signup',
    country            STRING            COMMENT 'Always India',
    email              STRING            COMMENT 'Email address',
    segment            STRING            COMMENT 'Premium / Standard / Basic / Trial',
    is_active          INT               COMMENT '1 = active  0 = inactive',
    pipeline_run_id    STRING            COMMENT 'RUN_ID that last updated this row'
)
USING DELTA
COMMENT 'Customer dimension — one row per customer, latest record from Bronze';

-- ────────────────────────────────────────────────────────────
-- FACT TABLES
-- ────────────────────────────────────────────────────────────

-- TABLE : customer360_silver.fact_transactions
-- GRAIN : One row per transaction (deduped on transaction_key)
-- NOTE  : net_amount = amount * (1 - discount_pct / 100)
--         All dimension keys resolved from lookup joins.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.fact_transactions (
    transaction_key  STRING  NOT NULL  COMMENT 'PK — same as transaction_id from Bronze',
    customer_key     STRING            COMMENT 'FK → dim_customer.customer_key',
    product_key      STRING            COMMENT 'FK → dim_product.product_key',
    date_key         STRING            COMMENT 'FK → dim_date.date_key — YYYY-MM-DD of transaction',
    location_key     STRING            COMMENT 'FK → dim_location.location_key — LOC_UNKNOWN if unmapped',
    payment_key      STRING            COMMENT 'FK → dim_payment_method.payment_key — PAY_UNKNOWN if unmapped',
    gross_amount     DOUBLE            COMMENT 'Original transaction amount in INR',
    net_amount       DOUBLE            COMMENT 'Amount after discount — gross * (1 - discount_pct/100)',
    discount_pct     INT               COMMENT '0 / 5 / 10 / 15 / 20',
    status           STRING            COMMENT 'success / failed / refunded',
    platform         STRING            COMMENT 'iOS / Android / Web',
    category         STRING            COMMENT 'Denormalised category for convenience',
    pipeline_run_id  STRING            COMMENT 'RUN_ID that created this row'
)
USING DELTA
COMMENT 'Transaction fact — one row per transaction with 5 dimension keys';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.fact_sessions
-- GRAIN : One row per app session (deduped on session_key)
-- NOTE  : location_key resolved via customer home city.
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.fact_sessions (
    session_key            STRING  NOT NULL  COMMENT 'PK — same as session_id from Bronze',
    customer_key           STRING            COMMENT 'FK → dim_customer.customer_key',
    date_key               STRING            COMMENT 'FK → dim_date.date_key — date of session start',
    location_key           STRING            COMMENT 'FK → dim_location.location_key — resolved via customer city',
    session_duration_mins  DOUBLE            COMMENT 'Session length in minutes',
    pages_visited          INT               COMMENT 'Pages or screens visited in session',
    actions_taken          INT               COMMENT 'Clicks, searches, adds to cart etc',
    device_type            STRING            COMMENT 'Mobile / Desktop / Tablet',
    platform               STRING            COMMENT 'iOS / Android / Web',
    is_bounce              INT               COMMENT '1 = session under 60 seconds  0 = normal',
    pipeline_run_id        STRING            COMMENT 'RUN_ID that created this row'
)
USING DELTA
COMMENT 'Session fact — one row per app session with engagement metrics';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_silver.fact_support
-- GRAIN : One row per support ticket (deduped on ticket_key)
-- NOTE  : is_resolved derived from status == closed
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_silver.fact_support (
    ticket_key          STRING  NOT NULL  COMMENT 'PK — same as ticket_id from Bronze',
    customer_key        STRING            COMMENT 'FK → dim_customer.customer_key',
    date_key            STRING            COMMENT 'FK → dim_date.date_key — date ticket was created',
    issue_type          STRING            COMMENT 'Payment Failure / Delivery Issue / Login Problem / Refund Request / Product Defect / Account Query',
    priority            STRING            COMMENT 'Low / Medium / High / Critical',
    status              STRING            COMMENT 'open / closed',
    is_resolved         INT               COMMENT '1 = closed  0 = open — derived from status',
    resolution_hours    DOUBLE            COMMENT 'Hours from creation to resolution — NULL if open',
    satisfaction_score  INT               COMMENT 'Customer rating 1-5 — NULL if open',
    pipeline_run_id     STRING            COMMENT 'RUN_ID that created this row'
)
USING DELTA
COMMENT 'Support fact — one row per ticket with resolution and satisfaction data';